In [ ]:
import pandas as pd
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import KeyBERTInspired
import re
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random
import os
import requests
from PIL import Image
from urllib.parse import urlparse
import hashlib
from torch import bfloat16
import transformers
import openai
from bertopic.representation import OpenAI

image_folder = 'images'
os.makedirs(image_folder, exist_ok=True)

updated_images = []
for i, image_url in enumerate(images):
    if image_url is not None and isinstance(image_url, str):
        try:
            # Generate a unique filename based on the URL content for consistency and avoid duplicates
            response = requests.get(image_url, stream=True, timeout=10)
            response.raise_for_status() # Raise an exception for bad status codes

            # Use a hash of the URL as part of the filename to ensure uniqueness
            url_hash = hashlib.md5(image_url.encode('utf-8')).hexdigest()
            # Try to get the file extension from the URL, or default to .jpg
            path = urlparse(image_url).path
            ext = os.path.splitext(path)[1]
            if not ext:
                ext = '.jpg' # Default extension if none found

            filename = f"{url_hash}{ext}"
            local_image_path = os.path.join(image_folder, filename)

            # Save image content
            with open(local_image_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            updated_images.append(local_image_path)
        except requests.exceptions.RequestException as e:
            print(f"Could not download image from {image_url}: {e}")
            updated_images.append(None)
        except Exception as e:
            print(f"An error occurred processing image {image_url}: {e}")
            updated_images.append(None)
    else:
        updated_images.append(None)

images = updated_images
print(f"Downloaded {len([img for img in images if img is not None])} images to /{image_folder}.")
print("First 10 updated image paths:")
print(images[:10])